In [3]:
import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import re
from sklearn.metrics import r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.gaussian_process.kernels import WhiteKernel
from sklearn.gaussian_process.kernels import ExpSineSquared
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

import tensorflow as tf
from keras import layers

import pickle 


# Pickle Save object
def save_obj(obj, filename):
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'wb') as outp:  # Overwrites any existing file.
        pickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)

# Pickle Read Object
def read_obj(filename):
    
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'rb') as inp:  # Reads File
        return pickle.load(inp)

In [3]:
# install tensorflow: https://caffeinedev.medium.com/how-to-install-tensorflow-on-m1-mac-8e9b91d93706

In [4]:
################ Function getAPIData  ####################
##  sub function to import data from skyspark web data grabber
##  inputs: A json object with method, endpoint, date range, interval, and fold method
##  outputs: A DataFrame object
########################################################
def getAPiData_Test(inputs):
    # data url and retrival request
    url = "https://fm-energywatch.ad.gatech.edu/skysparkBackend/"

    try:
        r = requests.post(url, json=inputs)
    except:
        'An Error Occured. Are you connected to the GT Internet Network (e.g: GT VPN)'

    # metadata variable extractor
    meta = r.text.split('\n')[1]
    unit = re.search('unit:\"(.+?)\"',meta).group(1)
    kind = re.search('kind:\"(.+?)\"',meta).group(1)

    # extract values -> split into dates and values -> assign as dataframe
    v = r.text.split('\n')[2:-2]
    v_dates = pd.DataFrame([value.split(',')[0] for value in v])
    v_values = pd.DataFrame([value.split(',')[1][:-len(unit)] for value in v])

    #concat Date and Data -> set column names -> set date as index
    output = pd.concat([v_dates, v_values], axis=1, join='inner')
    output.columns = ['Date/Time', 'Values']
    output = output.set_index('Date/Time')

    # adjust datetime for Daylight savings, remove localization info to standardize data
    output.index = pd.to_datetime(output.index, format='%Y-%m-%dT%H:%M:%S%z New_York', utc = True)
    output.index = pd.to_datetime(output.index, format='%Y-%m-%d %H:%M:%S+00:00').tz_convert(tz = 'EST')
    output.index = output.index.tz_localize(None)


    # getting values, removing units, and changing to float if number
    if kind == 'Number': # changing string values to float if metadata label = number
        output['Values'] = pd.to_numeric(output['Values'])
    
    return output, meta

In [5]:
def getAPiData(inputs):
    # data url and retrival request
    url = "https://fm-energywatch.ad.gatech.edu/skysparkBackend/"

    try:
        r = requests.post(url,json=inputs)
        print(r)
    except:
        'An Error Occured. Are you connected to the GT Internet Network (e.g: GT VPN)'

    # metadata variable extractor
    meta = r.text.split('\n')[1]
    unit = re.search('unit:\"(.+?)\"',meta).group(1)
    kind = re.search('kind:\"(.+?)\"',meta).group(1)

    # extract values -> split into dates and values -> assign as dataframe
    v = r.text.split('\n')[2:-2]
    v_dates = pd.DataFrame([value.split(',')[0] for value in v])
    v_values = pd.DataFrame([value.split(',')[1][:-len(unit)] for value in v])

    #concat Date and Data -> set column names -> set date as index
    output = pd.concat([v_dates, v_values], axis=1, join='inner')
    output.columns = ['Date/Time', 'Values']
    output = output.set_index('Date/Time')

    # adjust datetime for Daylight savings, remove localization info to standardize data
    output.index = pd.to_datetime(output.index, format='%Y-%m-%dT%H:%M:%S%z New_York', utc = True)
    output.index = pd.to_datetime(output.index, format='%Y-%m-%d %H:%M:%S+00:00').tz_convert(tz = 'EST')
    output.index = output.index.tz_localize(None)


    # getting values, removing units, and changing to float if number
    if kind == 'Number': # changing string values to float if metadata label = number
        output['Values'] = pd.to_numeric(output['Values'])
    
    return output

In [6]:
################ Function daylightSavingAdj #############
##  Function daylightSavingAdj(inputs):
##  sub function to adjust daylight savings time periods
##  inputs: A time series dataFrame
##  outputs: adjusted dataframe
########################################################
def daylightSavingAdj(input_df):
    # drops first line if it includes previous date (due to daylight savings)
    if input_df.index[0].hour > 1:
            input_df =  input_df.drop(input_df.index[[0]])

    #copies last data point if it is missing last data point (due to daylight savings)
    if input_df.index[-1].hour == 22 :
        new_date = dt.datetime(input_df.index[-1].year,input_df.index[-1].month,input_df.index[-1].day,23,0,0)
        append_row = pd.DataFrame(input_df[-1:].values, index=[new_date], columns=input_df.columns)
        input_df = pd.concat([input_df, append_row], axis = 0)
        
    return input_df

In [7]:
################ Function removeSpike ###################
##  Function removeSpike(inputs, name, spike, avgSpike):
##  sub function to remove spikes in Data
##  inputs: A time series dataFrame
##  outputs: adjusted dataframe
#########################################################
def removeSpike(df, spike, avgSpike):

    dropRows = []
    intVal = df.iloc[0]
    
    for x in range(1,len(df.index)-1):
        val = df.iloc[x]
        if abs(val - intVal) > spike:
            dropRows.append(x)
        elif x > 24:
            avgVal = np.mean(df.iloc[x-24:x])
            if abs(avgVal - intVal) > avgSpike:
                dropRows.append(x)

    df.drop(df.index[dropRows], axis=0, inplace = True)

    df = df.resample('H').mean()
    df = df.interpolate(method='linear')

    return df

In [11]:
def getData_Add(startDate, endDate): # querying data from skySpark API
    dateRange = startDate + ".." + endDate
    
    # howey
    b081_pt1 = "@p:fm_meters:r:2af29725-49eb4b55"
    b081_pt2    = '@p:fm_meters:r:2af297b5-078590ca'
    b081_raw_pt1 = "@p:fm_meters:r:25c061b7-39e5ea76"
    b081_raw_pt2 = "@p:fm_meters:r:2af29761-5186b1f0"
    b081_rt_pt1 = "@p:fm_meters:r:25c061b7-448fc89c"
    b081_rt_pt2 = "@p:fm_meters:r:25c061b7-777f6668"
    b081_st_pt1 = "@p:fm_meters:r:25c061b7-ecefaa7d"
    b081_st_pt2 = "@p:fm_meters:r:25c061b7-e64891f8"
    b081_fw_pt1 = "@p:fm_meters:r:25c07a50-7ddf84dc"
    b081_fw_pt2 = "@p:fm_meters:r:25c061b7-b91da08f"
    
    # Instructional center
    b055_pt = "@p:fm_meters:r:2af28778-5c770174"
    b055_raw_pt = "@p:fm_meters:r:25b89942-34a2b165"
    b055_rt_pt = "@p:fm_meters:r:25b89942-c1d6a0cf"
    b055_st_pt = "@p:fm_meters:r:25b89942-05bd2994"
    b055_fw_pt = "@p:fm_meters:r:25b89942-8f00a93b"
    
    # Bogg
    b103_pt = "@p:fmc  xcxcc cx_meters:r:2af1e45c-b90fcdf2"
    b103_raw_pt = "@p:fm_meters:r:26628b6f-135056a9"
    b103_rt_pt = "@p:fm_meters:r:26628b6f-77f90fdf"
    b103_st_pt = "@p:fm_meters:r:26628b6f-fc73ec8b"
    b103_fw_pt = "@p:fm_meters:r:26628b6f-9536b1e5"

    # culc
    b166_pt = "@p:fm_meters:r:2af2e9f2-b3d037a1"
    b166_raw_pt = "@p:fm_meters:r:265fdd65-4bed65a9"
    b166_rt_pt = "@p:fm_meters:r:25e2cbd8-fa754a86"
    b166_st_pt = "@p:fm_meters:r:25e2cbd8-2ef7af71"
    b166_fw_pt = "@p:fm_meters:r:25e2cbd8-c490c48e"
    
    # skiles (check again)
    b002_pt = "@p:fm_meters:r:2afeece6-7c68a9cd"
    b002_raw_pt = "@p:fm_meters:r:25af3388-c2542c5a"
    b002_rt_pt = "@p:fm_meters:r:25af3388-3e360aa7"
    b002_st_pt = "@p:fm_meters:r:25af3388-9fbc1450"
    b002_fw_pt = "@p:fm_meters:r:25af3388-4753e56c"

    # mse
    b084_pt = "@p:fm_meters:r:2af1de7e-42cd89b9"
    b084_raw_pt = "@p:fm_meters:r:25c07b8f-90058269"
    b084_rt_pt = "@p:fm_meters:r:25c07b8f-7e07bf7f"
    b084_st_pt = "@p:fm_meters:r:25c07b8f-b6a4f08e"
    b084_fw_pt = "@p:fm_meters:r:25c07b8f-8cc7e02d"
    
    
    # Default settings for data retrieval
    mthd = 'getData'
    date = dateRange
    rolp = 'avg'
    intv = '1hr'
    pd.set_option('display.max_rows', 100)

    # Retrive Data from cloud backend
    b081_1    = getAPiData({'method': mthd, 'endpoint': \
        b081_pt1, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_2    = getAPiData({'method': mthd, 'endpoint': \
            b081_pt2, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_raw_1    = getAPiData({'method': mthd, 'endpoint': \
            b081_raw_pt1, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_raw_2    = getAPiData({'method': mthd, 'endpoint': \
           b081_raw_pt2, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_rt_1 = getAPiData({'method': mthd, 'endpoint':\
         b081_rt_pt1, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_rt_2 = getAPiData({'method': mthd, 'endpoint':\
         b081_rt_pt2, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_st_1    = getAPiData({'method': mthd, 'endpoint': \
                b081_st_pt1, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_st_2    = getAPiData({'method': mthd, 'endpoint': \
           b081_st_pt2, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_fw_1 = getAPiData({'method': mthd, 'endpoint':\
         b081_fw_pt1, 'date': date, 'rollup': rolp, 'interval': intv})
    b081_fw_2 = getAPiData({'method': mthd, 'endpoint':\
         b081_fw_pt2, 'date': date, 'rollup': rolp, 'interval': intv})
    b055    = getAPiData({'method': mthd, 'endpoint': \
        b055_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b055_raw    = getAPiData({'method': mthd, 'endpoint': \
            b055_raw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b055_rt    = getAPiData({'method': mthd, 'endpoint': \
            b055_rt_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b055_st    = getAPiData({'method': mthd, 'endpoint': \
           b055_st_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b055_fw = getAPiData({'method': mthd, 'endpoint':\
         b055_fw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b103 = getAPiData({'method': mthd, 'endpoint':\
         b103_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b103_raw = getAPiData({'method': mthd, 'endpoint':\
         b103_raw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b103_rt    = getAPiData({'method': mthd, 'endpoint': \
                b103_rt_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b103_st    = getAPiData({'method': mthd, 'endpoint': \
           b103_st_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b103_fw = getAPiData({'method': mthd, 'endpoint':\
         b103_fw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b166    = getAPiData({'method': mthd, 'endpoint': \
        b166_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b166_raw    = getAPiData({'method': mthd, 'endpoint': \
            b166_raw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b166_rt    = getAPiData({'method': mthd, 'endpoint': \
            b166_rt_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b166_st    = getAPiData({'method': mthd, 'endpoint': \
           b166_st_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b166_fw = getAPiData({'method': mthd, 'endpoint':\
         b166_fw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b002 = getAPiData({'method': mthd, 'endpoint':\
         b002_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b002_raw    = getAPiData({'method': mthd, 'endpoint': \
                b002_raw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b002_rt    = getAPiData({'method': mthd, 'endpoint': \
           b002_rt_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b002_st = getAPiData({'method': mthd, 'endpoint':\
         b002_st_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b002_fw = getAPiData({'method': mthd, 'endpoint':\
         b002_fw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b084    = getAPiData({'method': mthd, 'endpoint': \
        b084_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b084_raw    = getAPiData({'method': mthd, 'endpoint': \
            b084_raw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b084_rt    = getAPiData({'method': mthd, 'endpoint': \
            b084_rt_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b084_st    = getAPiData({'method': mthd, 'endpoint': \
           b084_st_pt, 'date': date, 'rollup': rolp, 'interval': intv})
    b084_fw = getAPiData({'method': mthd, 'endpoint':\
         b084_fw_pt, 'date': date, 'rollup': rolp, 'interval': intv})
 
     #combine data into two dataframes. One fo r operation, one for response
    data = pd.concat([b081_1, b081_2, b081_raw_1, b081_raw_2, b081_rt_1, b081_rt_2, b081_st_1, b081_st_2, b081_fw_1, b081_fw_2, b055, b055_raw, b055_rt, b055_st, b055_fw, b103, b103_raw, b103_rt, b103_st, b103_fw, b166, b166_raw, b166_rt, b166_st, b166_fw, b002, b002_raw, b002_rt, b002_st, b002_fw, b084, b084_raw, b084_rt, b084_st, b084_fw], axis = 1, join = "inner")
    data.columns = ['b081_1', "b081_2", "b081_raw_1", "b081_raw_2", "b081_rt_1", "b081_rt_2", "b081_st_1", "b081_st_2", "b081_fw_1", "b081_fw_2", "b055", "b055_raw", "b055_rt", "b055_st", "b055_fw", "b103", "b103_raw", "b103_rt", "b103_st", "b103_fw", "b166", "b166_raw", "b166_rt", "b166_st", "b166_fw", "b002", "b002_raw", "b002_rt", "b002_st", "b002_fw", "b084", "b084_raw", "b084_rt", "b084_st", "b084_fw"]

    # Daylight Saving Adjustment. Standardized to EST
    # data = daylightSavingAdj(data)
    
    return data

In [12]:
def getData_CHW_Energy(startDate, endDate):
    dateRange = startDate + ".." + endDate

    # Define endpoints for all buildings
    building_endpoints = {
        "b181": [
            "@p:fm_meters:r:2af2edd3-89dda5ad",
            "@p:fm_meters:r:25d2d013-812dae1a",
            "@p:fm_meters:r:25d2d013-5bad37c4",
            "@p:fm_meters:r:25d2d013-71eb4b05",
            "@p:fm_meters:r:2662b430-56faa689"
        ],
   
        "b146": [
            "@p:fm_meters:r:2af2e68c-876a164a",
            "@p:fm_meters:r:25d2e873-1d97bc34",
            "@p:fm_meters:r:25d2e873-55984f5e",
            "@p:fm_meters:r:25d2e873-dbded0a1",
            "@p:fm_meters:r:25d2e873-d0e62c3a"
        ],
        "b153": [
            "@p:fm_meters:r:2af2e933-a1a6a6a0",
            "@p:fm_meters:r:25d2e380-ca5b3de8",
            "@p:fm_meters:r:25d2e380-420f1ae4",
            "@p:fm_meters:r:25d2e380-a8066188",
            "@p:fm_meters:r:25d2e380-58bfa031"
        ],
        "b147": [
            "@p:fm_meters:r:2af2e85a-c9bf45de",
            "@p:fm_meters:r:25d2e63b-cb579645",
            "@p:fm_meters:r:25d2e63b-b9bf9fbb",
            "@p:fm_meters:r:25d2e63b-5a17a83a",
            "@p:fm_meters:r:25d2e63b-cda1afe0"
        ],
        "b165": [
            "@p:fm_meters:r:2af2e9f2-126fb4bf",
            "@p:fm_meters:r:25d2e292-0a6e14a7",
            "@p:fm_meters:r:25d2e292-d0bcb675",
            "@p:fm_meters:r:25d2e292-b7f48ec6",
            "@p:fm_meters:r:25d2e292-e390779f"
        ],
        "b167": [
            "@p:fm_meters:r:2af2ead8-cc863fcf",
            "@p:fm_meters:r:271081d8-e815bb03",
            "@p:fm_meters:r:25e2badf-f8360e27",
            "@p:fm_meters:r:25e2badf-1e3a06cb",
            "@p:fm_meters:r:25e2badf-727a349e"
        ],
        "b195": [
            "@p:fm_meters:r:2af2f09d-979cd3aa",
            "@p:fm_meters:r:2662b51d-bf6c8bf2",
            "@p:fm_meters:r:2662b51d-fe46ff0b",
            "@p:fm_meters:r:2662b51d-883e5098",
            "@p:fm_meters:r:2662b51d-b27071cb"
        ],
        "b790": [
            "@p:fm_meters:r:2afd1809-84851016",
            "@p:fm_meters:r:2662cb0c-3dc0e78a",
            "@p:fm_meters:r:25d2d56f-9d1ccf90",
            "@p:fm_meters:r:25d2d56f-a4d86f94",
            "@p:fm_meters:r:25d2d56f-9c9a8238"
        ], 
      
        "b116": [
            "@p:fm_meters:r:2aeebde9-c9df7e36",
            "@p:fm_meters:r:25d2f0a8-e2fd5ee8",
            "@p:fm_meters:r:25d2f0a8-40cb8ec4",
            "@p:fm_meters:r:25d2f0a8-a6740d2f",
            "@p:fm_meters:r:25d2f0a8-37cc7546"
        ],
        "b064": [
            "@p:fm_meters:r:2af294b6-486b615a",
            "@p:fm_meters:r:25b89fe0-2328c6fe",
            "@p:fm_meters:r:25b89fe0-ff619018",
            "@p:fm_meters:r:25b89fe0-8963d48c",
            "@p:fm_meters:r:25b89fe0-de8204c1"
        ]
    }

    # Default settings for data retrieval
    mthd = 'getData'
    date = dateRange
    rolp = 'avg'
    intv = '1hr'
    pd.set_option('display.max_rows', 100)
    # building_names = ["b181", "b146", "b153", "b147", "b165", "b167", "b195", "b790", "b116", "b064"]

    # Initialize an empty DataFrame
    data = pd.DataFrame()
    
    # Iterate through building endpoints
    for building_name, endpoints in building_endpoints.items():
        for suffix in ["pt", "raw", "rt", "st", "fw"]:
            # Check if there are endpoints for the building
            if building_name in building_endpoints:
                print(building_name)
                # Retrieve Data from cloud backend
                endpoint = endpoints.pop(0)  # Take the first endpoint for the building
                building_data = getAPiData({'method': mthd, 'endpoint': endpoint, 'date': date, 'rollup': rolp, 'interval': intv})

                # Assign column names
                column_name = f"{building_name}_{suffix}"
                data[column_name] = building_data
    return data


In [13]:
"""
"b061": [
    "@p:fm_meters:r:2af29344-77e64f2e",
    "@p:fm_meters:r:26612ca6-aaed175a",
    "@p:fm_meters:r:26612ca6-e678cff5",
    "@p:fm_meters:r:26612ca6-3e7f2fb5",
    "@p:fm_meters:r:26612ca6-e188102c",
    "@p:fm_meters:r:26612ca6-d7d613b6",
    "@p:fm_meters:r:26612ca6-20990577"
],
        """   

'\n"b061": [\n    "@p:fm_meters:r:2af29344-77e64f2e",\n    "@p:fm_meters:r:26612ca6-aaed175a",\n    "@p:fm_meters:r:26612ca6-e678cff5",\n    "@p:fm_meters:r:26612ca6-3e7f2fb5",\n    "@p:fm_meters:r:26612ca6-e188102c",\n    "@p:fm_meters:r:26612ca6-d7d613b6",\n    "@p:fm_meters:r:26612ca6-20990577"\n],\n        '

In [14]:
df = getData_Add("2020-01-01", "2023-08-31")

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>


In [15]:
save_obj(df, "project_CHW_like_kendeda")

In [16]:
df.to_csv("./CSV data/project_CHW_like_kendeda.csv")

In [68]:
len(df.columns)

50

### Concat with kendeda

In [4]:
ken = pd.read_csv("./CSV data/Kendeda_20_23.csv")
df = pd.read_csv("./CSV data/project_CHW_like_kendeda.csv")

In [5]:
ken

,Unnamed: 0,WBT,RH,DBT,solarIrrad,Kendeda CHW,Kendeda CHW Tot,Kendeda CCHW Tot,Kendeda CCHW Ton,Kendeda CCHW RT,Kendeda CCHW ST,Kendeda CCHW F
0,2020-01-01 00:00:00,39.051667,59.433333,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01 01:00:00,38.154717,60.886792,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01 02:00:00,37.596429,62.589286,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01 03:00:00,37.000000,63.258621,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01 04:00:00,36.494231,63.826923,40.300000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
29251,2023-05-03 19:00:00,55.471667,35.666667,66.906667,3.85,10.01,124420.0,124420.0,7.176667,44.095000,43.462728,147.171818
29252,2023-05-03 20:00:00,54.411667,38.395833,65.171666,0.00,4.76,124430.0,124430.0,4.510000,44.364285,43.733751,111.796249
29253,2023-05-03 21:00:00,53.570000,41.116666,63.646667,0.00,5.61,124430.0,124430.0,1.295714,47.491428,47.986667,36.345000
29254,2023-05-03 22:00:00,52.796666,43.233334,62.226666,0.00,6.51,124430.0,124430.0,0.010000,49.027499,50.737499,0.000000


In [6]:
ken['Date/Time'] = ken['Unnamed: 0']

In [7]:
ken.columns

Index(['Unnamed: 0', 'WBT', 'RH', 'DBT', 'solarIrrad', 'Kendeda CHW',
       'Kendeda CHW Tot', 'Kendeda CCHW Tot', 'Kendeda CCHW Ton',
       'Kendeda CCHW RT', 'Kendeda CCHW ST', 'Kendeda CCHW F', 'Date/Time'],
      dtype='object')

In [8]:
ken = ken[['Kendeda CCHW Tot', 'Kendeda CCHW RT', 'Kendeda CCHW ST', 'Kendeda CCHW F', 'Date/Time']]

In [9]:
ken.columns = ['b210_pt', 'b210_rt', "b210_st", 'b210_fw', 'Date/Time']

In [10]:
final = ken.merge(df)

In [11]:
final.index = final['Date/Time']

In [12]:
final = final.drop(['Date/Time'], axis=1)

In [13]:
final.head()

,b210_pt,b210_rt,b210_st,b210_fw,b081_1,b081_2,b081_raw_1,b081_raw_2,b081_rt_1,b081_rt_2,...,b002,b002_raw,b002_rt,b002_st,b002_fw,b084,b084_raw,b084_rt,b084_st,b084_fw
Date/Time,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,308472.867188,NaN,308472.867188,NaN,55.450001,55.450001,...,2647859.00,2647859.00,49.650001,47.400002,201.325001,3788099.00,3788099.00,56.575000,47.500,74.774998
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,308480.296875,NaN,308480.296875,NaN,55.200000,55.200000,...,2647877.00,2647877.00,49.425000,47.475000,204.250000,3788127.25,3788127.25,57.300001,47.500,64.500000
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,308488.250000,NaN,308488.250000,NaN,54.525000,54.525000,...,2647892.75,2647892.75,49.349999,47.425001,205.524998,3788154.00,3788154.00,58.275000,47.500,49.725000
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,308496.046875,NaN,308496.046875,NaN,54.874999,54.874999,...,2647908.75,2647908.75,49.225000,47.400002,204.399998,3788180.75,3788180.75,56.950001,47.475,62.425000
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,308504.023438,NaN,308504.023438,NaN,55.924999,55.924999,...,2647924.50,2647924.50,49.299999,47.425001,198.875000,3788207.00,3788207.00,56.724999,47.500,56.000000


In [14]:
final.to_csv("./CSV data/Project_final_20_23.csv")
save_obj(final, "Project_final_20_23")